# Processing results from different VLMs

In [11]:
#imports
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


## Local VLMs for scenario prediction (from frames)

In [12]:
#go through all files that start with 'results_', then turn it into a pandas dataframe, 
# #then put everything into a single df where the prediction if the mode of the frames
def load_results(directory):
    results = pd.DataFrame()
    full_results = pd.DataFrame()
    for filename in os.listdir(directory):
        if filename.startswith('results_'):
            filepath = os.path.join(directory, filename)
            df = pd.read_csv(filepath)
            #add column in the beginning with the filename without 'results_' and '.csv'
            df['model'] = filename[8:-4]
            #make it first column
            df = df[['model'] + [col for col in df.columns if col != 'model']]

            #group by video_name and take the mode of the predictions
            df_grouped = df.groupby('video_name').agg(lambda x: x.mode()[0] if not x.mode().empty else np.nan).reset_index()
            #add the grouped dataframe to the full results
            full_results = pd.concat([full_results, df_grouped], ignore_index=True)

            results = pd.concat([results, df], ignore_index=True)

    #group by filename, by video, and take the mode of the predictions

    return results, full_results

In [13]:
path = '../results/'
results, full_results = load_results(path)

#remove NaN values from the results
print("Removing NaN values from the results...")
print(results.shape)
#drop rows if 'outcome_prediction' contains word "Error"
results = results[~results['outcome_prediction'].str.contains("Error", na=False)]
print(results.shape)
print(full_results.shape)
full_results = full_results[~full_results['outcome_prediction'].str.contains("Error", na=False)]
print("Full results after removing NaN values:")
print(full_results.shape)

#save the results to a csv file
results.to_csv('all_predictions.csv', index=False)
#save the full results to a csv file
full_results.to_csv('all_predictions_grouped.csv', index=False)

#see how many unique videos are for each model
unique_videos_per_model = results.groupby('model')['video_name'].nunique().reset_index()
print(unique_videos_per_model)

Removing NaN values from the results...
(5301, 4)
(5236, 4)
(270, 4)
Full results after removing NaN values:
(267, 4)
            model  video_name
0    deepseek_ocr          30
1          gemma3          30
2      gemma3_27b          30
3   llama32vision          30
4           llava          30
5     llavallama3          30
6  mistralsmall32          29
7          qwen25          30
8           qwen3          30


In [14]:
#now, per model, print how many videos are "Poorly" and how many are "Well", accounting for small variations in the writing
for model in full_results['model'].unique():
    print(f"Model: {model}")
    model_results = full_results[full_results['model'] == model]
    poorly_count = model_results['outcome_prediction'].str.contains('Poorly', case=False, na=False).sum()
    well_count = model_results['outcome_prediction'].str.contains('Well', case=False, na=False).sum()
    print(f"  Poorly: {poorly_count}")
    print(f"  Well: {well_count}")

    

Model: llama32vision
  Poorly: 30
  Well: 0
Model: llavallama3
  Poorly: 5
  Well: 25
Model: deepseek_ocr
  Poorly: 4
  Well: 26
Model: gemma3
  Poorly: 30
  Well: 0
Model: qwen25
  Poorly: 14
  Well: 16
Model: mistralsmall32
  Poorly: 26
  Well: 1
Model: qwen3
  Poorly: 7
  Well: 23
Model: gemma3_27b
  Poorly: 30
  Well: 0
Model: llava
  Poorly: 25
  Well: 5


In [15]:
#for outcome_prediction, replace poorly with 1 and well with 0
#if it contains "poorly", make new list with 1, if it contains "well", make new list with 0
full_results['outcome_prediction_numeric'] = full_results['outcome_prediction'].apply(
    lambda x: 1 if 'poorly' in str(x).lower() else (0 if 'well' in str(x).lower() else np.nan)
)
#save the full results with numeric outcome prediction to a csv file
full_results.to_csv('all_predictions_grouped.csv', index=False)


full_results


,video_name,model,frame,outcome_prediction,outcome_prediction_numeric
0,11_final.mp4,llama32vision,0,Poorly.,1
1,12_final.mp4,llama32vision,0,Poorly.,1
2,14_final.mp4,llama32vision,0,Poorly.,1
3,15_final.mp4,llama32vision,0,Poorly.,1
4,19_final.mp4,llama32vision,0,Poorly.,1
...,...,...,...,...,...
265,59_final.mp4,llava,0,Well,0
266,60_final.mp4,llava,0,Poorly,1
267,6_final.mp4,llava,0,Poorly,1
268,7_final.mp4,llava,0,Poorly,1


In [16]:
#now, get the groundtruth and see if they got it right
#open csv with columns Video,Question Mapping,Average Class of Human Predicion,True Outcome,

gt_df = pd.read_csv('../../../dataset_scenarios/analyze_predictions.csv')

#map each video to its true outcome
gt_df = gt_df[['Video', 'True Outcome']].rename(columns={'Video': 'video_name', 'True Outcome': 'true_outcome'})
#video are name without the .mp4 extension, so we need to add it to each one
gt_df['video_name'] = gt_df['video_name'].apply(lambda x: x + '.mp4' if not x.endswith('.mp4') else x)
#print type of true_outcome
gt_df['true_outcome'] = gt_df['true_outcome'].astype(int)
#dictionary to map video names to true outcomes
gt_dict = dict(zip(gt_df['video_name'], gt_df['true_outcome']))

#add the "true_outcome" column to the full_results dataframe
full_results['true_outcome'] = full_results['video_name'].map(gt_dict)

full_results


,video_name,model,frame,outcome_prediction,outcome_prediction_numeric,true_outcome
0,11_final.mp4,llama32vision,0,Poorly.,1,1
1,12_final.mp4,llama32vision,0,Poorly.,1,1
2,14_final.mp4,llama32vision,0,Poorly.,1,1
3,15_final.mp4,llama32vision,0,Poorly.,1,1
4,19_final.mp4,llama32vision,0,Poorly.,1,0
...,...,...,...,...,...,...
265,59_final.mp4,llava,0,Well,0,0
266,60_final.mp4,llava,0,Poorly,1,0
267,6_final.mp4,llava,0,Poorly,1,1
268,7_final.mp4,llava,0,Poorly,1,1


In [17]:
#models and metrics df
prediction_performance_df = pd.DataFrame(columns=['model', 'accuracy', 'precision', 'recall', 'f1_score', 'poorly_ratio'])


#now, go video by video and see if the model got it right
correct_predictions = []
for model in full_results['model'].unique():
    model_results = full_results[full_results['model'] == model]
    y_pred = model_results['outcome_prediction_numeric']
    y_true = []
    for index, row in model_results.iterrows():
        video_name = row['video_name']
        true_outcome = gt_dict.get(video_name, np.nan)
        y_true.append(true_outcome)

    
    y_pred = np.array(y_pred)
    y_true = np.array(y_true)
    #print(y_pred)
    #print(y_true)

    #get accuracy, precision, recall, f1 score
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    
    # Calculate poorly ratio (ratio of poorly/0 predictions to all predictions)
    poorly_count = (y_pred == 1).sum()  # 1 represents "poorly" in our encoding
    total_count = len(y_pred)
    poorly_ratio = poorly_count / total_count if total_count > 0 else 0

    print(f"Model: {model}")
    print(f"  Accuracy: {accuracy:.2f}")
    print(f"  Precision: {precision:.2f}")
    print(f"  Recall: {recall:.2f}")
    print(f"  F1 Score: {f1:.2f}")
    print(f"  Poorly Ratio: {poorly_ratio:.2f}")

    df_pred = pd.DataFrame({
        'model': [model],
        'accuracy': [accuracy],
        'precision': [precision],
        'recall': [recall],
        'f1_score': [f1],
        'poorly_ratio': [poorly_ratio]
    })
    prediction_performance_df = pd.concat([prediction_performance_df, df_pred], ignore_index=True)

#save the prediction performance to a csv file

prediction_performance_df.to_csv('prediction_performance.csv', index=False)
#save full results with true outcome to a csv file
full_results.to_csv('all_predictions_grouped_with_true_outcome.csv', index=False)

prediction_performance_df

Model: llama32vision
  Accuracy: 0.43
  Precision: 0.43
  Recall: 1.00
  F1 Score: 0.60
  Poorly Ratio: 1.00
Model: llavallama3
  Accuracy: 0.53
  Precision: 0.40
  Recall: 0.15
  F1 Score: 0.22
  Poorly Ratio: 0.17
Model: deepseek_ocr
  Accuracy: 0.50
  Precision: 0.25
  Recall: 0.08
  F1 Score: 0.12
  Poorly Ratio: 0.13
Model: gemma3
  Accuracy: 0.43
  Precision: 0.43
  Recall: 1.00
  F1 Score: 0.60
  Poorly Ratio: 1.00
Model: qwen25
  Accuracy: 0.50
  Precision: 0.43
  Recall: 0.46
  F1 Score: 0.44
  Poorly Ratio: 0.47
Model: mistralsmall32
  Accuracy: 0.52
  Precision: 0.50
  Recall: 1.00
  F1 Score: 0.67
  Poorly Ratio: 0.96
Model: qwen3
  Accuracy: 0.47
  Precision: 0.29
  Recall: 0.15
  F1 Score: 0.20
  Poorly Ratio: 0.23
Model: gemma3_27b
  Accuracy: 0.43
  Precision: 0.43
  Recall: 1.00
  F1 Score: 0.60
  Poorly Ratio: 1.00
Model: llava
  Accuracy: 0.33
  Precision: 0.36
  Recall: 0.69
  F1 Score: 0.47
  Poorly Ratio: 0.83


/tmp/ipykernel_3102682/3701129753.py:48: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  prediction_performance_df = pd.concat([prediction_performance_df, df_pred], ignore_index=True)


,model,accuracy,precision,recall,f1_score,poorly_ratio
0,llama32vision,0.433333,0.433333,1.000000,0.604651,1.000000
1,llavallama3,0.533333,0.400000,0.153846,0.222222,0.166667
2,deepseek_ocr,0.500000,0.250000,0.076923,0.117647,0.133333
3,gemma3,0.433333,0.433333,1.000000,0.604651,1.000000
4,qwen25,0.500000,0.428571,0.461538,0.444444,0.466667
5,mistralsmall32,0.518519,0.500000,1.000000,0.666667,0.962963
6,qwen3,0.466667,0.285714,0.153846,0.200000,0.233333
7,gemma3_27b,0.433333,0.433333,1.000000,0.604651,1.000000
8,llava,0.333333,0.360000,0.692308,0.473684,0.833333


In [19]:
# ===================================================================
# Fix Video Name Mapping Issue
# ===================================================================
print("\n" + "="*50)
print("Fixing Video Name Mapping")
print("="*50)

# Read the ground truth file again to get the proper mapping
gt_df_full = pd.read_csv('../../../dataset_scenarios/analyze_predictions.csv')

# Create mapping from question format (q_X) to video format (X_final.mp4)
question_to_video = dict(zip(gt_df_full['Question Mapping'], gt_df_full['Video']))
print("Question to Video mapping (sample):")
for i, (q, v) in enumerate(list(question_to_video.items())[:5]):
    print(f"   {q} -> {v}")

# Create the reverse mapping: video format to question format
video_to_question = {v + '.mp4': q for q, v in question_to_video.items()}
print("\nVideo to Question mapping (sample):")
for i, (v, q) in enumerate(list(video_to_question.items())[:5]):
    print(f"   {v} -> {q}")

# Now update the human data to use the correct video names for comparison
print("\n6. Testing the fixed mapping:")
print("   Sample model videos:", list(full_results['video_name'].unique())[:3])
print("   Corresponding question IDs:", [video_to_question.get(v, "NOT FOUND") for v in list(full_results['video_name'].unique())[:3]])




Fixing Video Name Mapping
Question to Video mapping (sample):
   q_2 -> 6_final
   q_3 -> 7_final
   q_4 -> 9_final
   q_5 -> 11_final
   q_6 -> 12_final

Video to Question mapping (sample):
   6_final.mp4 -> q_2
   7_final.mp4 -> q_3
   9_final.mp4 -> q_4
   11_final.mp4 -> q_5
   12_final.mp4 -> q_6

6. Testing the fixed mapping:
   Sample model videos: ['11_final.mp4', '12_final.mp4', '14_final.mp4']
   Corresponding question IDs: ['q_5', 'q_6', 'q_7']


In [20]:
# ===================================================================
# Human vs Model Predictions Analysis (adapted from online analysis)
# ===================================================================

print("\n" + "="*50)
print("Individual Human Predictions vs True Outcome")
print("="*50)       

# Load human predictions data
human_df = pd.read_csv('../../../dataset_scenarios/badidea_ground_truth.csv')

# Filter to only include specific participant IDs
valid_participant_ids = [1048, 1251, 1483, 1676, 2103, 2313, 2698, 2946, 3157, 3203, 3339, 3882, 5009, 5099, 5124, 5233, 5310, 6488, 7136, 7782, 7797, 8184, 8436, 8758, 8786, 9055, 9385, 9777, 9941]
print(f"Filtering to only include {len(valid_participant_ids)} specific participants")
print(f"Original dataset size: {len(human_df)}")
human_df = human_df[human_df['participant_id'].isin(valid_participant_ids)]
print(f"Filtered dataset size: {len(human_df)}")

# Use the correct mapping from the previous cell (question format to video format)
# Create a mapping from question ID to true outcome using the ground truth data
gt_df_full = pd.read_csv('../../../dataset_scenarios/analyze_predictions.csv')
question_to_outcome = dict(zip(gt_df_full['Question Mapping'], gt_df_full['True Outcome']))

print(f"Created question to outcome mapping with {len(question_to_outcome)} entries")
print("Sample mappings:", dict(list(question_to_outcome.items())[:5]))

# Get unique participant IDs (now filtered)
participant_ids = human_df['participant_id'].unique()
print(f"Number of participants after filtering: {len(participant_ids)}")

# Define metrics calculation function
def calculate_metrics(y_true, y_pred_prob):
    from sklearn.metrics import roc_auc_score

    # Convert to binary if needed
    y_pred = (np.array(y_pred_prob) >= 0.5).astype(int)
    y_true = np.array(y_true).astype(int)

    # Handle cases where only one class is present
    try:
        auc = roc_auc_score(y_true, y_pred_prob)
    except:
        auc = np.nan

    # Calculate metrics
    metrics = {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'auc': auc,
        'mse': np.mean((y_true - y_pred_prob) ** 2),
        'mae': np.mean(np.abs(y_true - y_pred_prob))
    }
    return metrics

# Calculate metrics for each individual human participant
human_metrics = {}
for participant_id in participant_ids:
    # Get this participant's predictions
    participant_data = human_df[human_df['participant_id'] == participant_id]
    print(f"Processing participant: {participant_id}")

    # Create lists to store true outcomes and participant predictions
    true_outcomes = []
    participant_predictions = []

    # Match participant responses with true outcomes using question IDs
    for _, row in participant_data.iterrows():
        question_id = row['response_video']  # This is already in q_X format
        
        if question_id in question_to_outcome:
            true_outcomes.append(question_to_outcome[question_id])
            participant_predictions.append(row['class'])

    # Calculate metrics if this participant has matching predictions
    if len(true_outcomes) > 0:
        print(f"   Found {len(true_outcomes)} matching predictions for participant {participant_id}")
        human_metrics[participant_id] = calculate_metrics(
            np.array(true_outcomes),
            np.array(participant_predictions)
        )
    else:
        print(f"   No matching predictions found for participant {participant_id}")

print(f"\nProcessed {len(human_metrics)} participants with valid metrics")

# Calculate average metrics across all participants
avg_human_metrics = {}
std_human_metrics = {}
for metric in ['accuracy', 'precision', 'recall', 'f1', 'auc', 'mse', 'mae']:
    values = [m[metric] for m in human_metrics.values() if not np.isnan(m[metric])]
    avg_human_metrics[metric] = np.mean(values) if values else np.nan
    std_human_metrics[metric] = np.std(values) if values else np.nan

# Display average human performance
if avg_human_metrics:
    human_df_metrics = pd.DataFrame(avg_human_metrics, index=['Average Individual Human'])
    print("\nAverage Individual Human Performance:")
    print(human_df_metrics.round(3))

    # Display standard deviation of human performance
    human_df_std = pd.DataFrame(std_human_metrics, index=['Standard Deviation'])
    print("\nStandard Deviation of Human Performance:")
    print(human_df_std.round(3))

    # Also show distribution of human performance
    print("\nDistribution of individual human accuracy:")
    human_accuracies = [metrics['accuracy'] for metrics in human_metrics.values()]
    if human_accuracies:
        print(f"Min: {min(human_accuracies):.3f}, Max: {max(human_accuracies):.3f}, Mean: {np.mean(human_accuracies):.3f}, Median: {np.median(human_accuracies):.3f}")
else:
    print("No valid human metrics calculated!")


Individual Human Predictions vs True Outcome
Filtering to only include 29 specific participants
Original dataset size: 865
Filtered dataset size: 865
Created question to outcome mapping with 30 entries
Sample mappings: {'q_2': 1, 'q_3': 1, 'q_4': 1, 'q_5': 1, 'q_6': 1}
Number of participants after filtering: 29
Processing participant: 1048
   Found 30 matching predictions for participant 1048
Processing participant: 1251
   Found 30 matching predictions for participant 1251
Processing participant: 1483
   Found 28 matching predictions for participant 1483
Processing participant: 1676
   Found 30 matching predictions for participant 1676
Processing participant: 2103
   Found 30 matching predictions for participant 2103
Processing participant: 2313
   Found 30 matching predictions for participant 2313
Processing participant: 2698
   Found 30 matching predictions for participant 2698
Processing participant: 2946
   Found 30 matching predictions for participant 2946
Processing participant

In [21]:
# ===================================================================
# Local Models vs Individual Human Predictions (FIXED)
# ===================================================================
print("\n" + "="*50)
print("Local Models vs Individual Human Predictions")
print("="*50)

# Use the video_to_question mapping from the previous cell
# Create a mapping from video name to model predictions
video_to_model_preds = {}
for model in full_results['model'].unique():
    model_data = full_results[full_results['model'] == model]
    # Remove NaN values from model predictions
    model_data = model_data.dropna(subset=['outcome_prediction_numeric'])
    video_to_model_preds[model] = dict(zip(model_data['video_name'], model_data['outcome_prediction_numeric']))
    print(f"Model {model}: {len(video_to_model_preds[model])} valid predictions")

# Calculate metrics for each model compared to each participant
model_vs_human_metrics = {}

for model in full_results['model'].unique():
    model_vs_human_metrics[model] = {}
    print(f"\nProcessing model: {model}")
    
    for participant_id in participant_ids:
        # Get this participant's predictions
        participant_data = human_df[human_df['participant_id'] == participant_id]
        
        # Create lists to store participant predictions and model predictions
        human_preds = []
        model_preds = []
        
        # Match participant responses with model predictions using the mapping
        for _, row in participant_data.iterrows():
            question_id = row['response_video']  # e.g., 'q_2'
            
            # Convert question ID to video name using the mapping
            if question_id in question_to_video:
                video_name = question_to_video[question_id] + '.mp4'  # Convert to video format
                
                # Check if this video has a model prediction
                if video_name in video_to_model_preds[model]:
                    model_pred = video_to_model_preds[model][video_name]
                    if not np.isnan(model_pred):
                        human_preds.append(row['class'])
                        model_preds.append(model_pred)
        
        # Calculate metrics if this participant has matching predictions
        if len(human_preds) > 0:
            print(f"   Participant {participant_id}: {len(human_preds)} matching predictions")
            try:
                model_vs_human_metrics[model][participant_id] = calculate_metrics(
                    np.array(human_preds),
                    np.array(model_preds)
                )
            except Exception as e:
                print(f"   Error calculating metrics for participant {participant_id}: {e}")
        else:
            print(f"   Participant {participant_id}: No matching predictions")

# Calculate average metrics for each model vs humans
avg_model_vs_human = {}
std_model_vs_human = {}
for model, participant_metrics in model_vs_human_metrics.items():
    if participant_metrics:  # Only if there are metrics for this model
        avg_model_vs_human[model] = {}
        std_model_vs_human[model] = {}
        for metric in ['accuracy', 'precision', 'recall', 'f1', 'auc', 'mse', 'mae']:
            values = [m[metric] for m in participant_metrics.values() if not np.isnan(m[metric])]
            avg_model_vs_human[model][metric] = np.mean(values) if values else np.nan
            std_model_vs_human[model][metric] = np.std(values) if values else np.nan

# Convert to DataFrame for easier comparison
if avg_model_vs_human:
    avg_model_vs_human_df = pd.DataFrame(avg_model_vs_human).T
    print("\n" + "="*50)
    print("RESULTS: Average Agreement with Individual Humans")
    print("="*50)
    print(avg_model_vs_human_df.round(3))
    
    std_model_vs_human_df = pd.DataFrame(std_model_vs_human).T
    print("\nStandard Deviation of Agreement with Individual Humans:")
    print(std_model_vs_human_df.round(3))
    
    # Save results
    avg_model_vs_human_df.to_csv('model_vs_human_agreement.csv')
    print("\nResults saved to 'model_vs_human_agreement.csv'")
else:
    print("\n❌ No valid metrics calculated - check data alignment issues")


Local Models vs Individual Human Predictions
Model llama32vision: 30 valid predictions
Model llavallama3: 30 valid predictions
Model deepseek_ocr: 30 valid predictions
Model gemma3: 30 valid predictions
Model qwen25: 30 valid predictions
Model mistralsmall32: 27 valid predictions
Model qwen3: 30 valid predictions
Model gemma3_27b: 30 valid predictions
Model llava: 30 valid predictions

Processing model: llama32vision
   Participant 1048: 30 matching predictions
   Participant 1251: 30 matching predictions
   Participant 1483: 28 matching predictions
   Participant 1676: 30 matching predictions
   Participant 2103: 30 matching predictions
   Participant 2313: 30 matching predictions
   Participant 2698: 30 matching predictions
   Participant 2946: 30 matching predictions
   Participant 3157: 30 matching predictions
   Participant 3203: 30 matching predictions
   Participant 3339: 30 matching predictions
   Participant 3882: 30 matching predictions
   Participant 5009: 30 matching predi